# 04 Model 2: RNN + Classifier
Training of architecture 2: the same frozen word embeddings as in model 1, a 2-layer RNN (LSTM or GRU) and a 2-layer classifier (ReLU non-linearity). The RNN and the classifier are trained. Experiments are tracked with Weights & Biases.

**In short.** Question: does an RNN encoder beat averaging in the same fixed-position format, and which settings work best? Answer: by the validation loss yes, in this format, and over 5 seeds also by the accuracy; `04b_per_choice.ipynb` tests another input format for model 1. Chosen: GRU with layer norm in the classifier, RNN hidden size 128, learning rate 0.001, dropout 0.5, batch size 128, best epoch 6 (validation loss 0.5867, accuracy 66.70%).

**GPU version (1.11):** run on a Tesla T4 on Google Colab, with deterministic mode, patience 10, batch size 128 from a test (section "Batch size"), 5 seeds, and the vocabulary of `02_preprocessing.ipynb`, which also holds the validation words with a GloVe vector. The matrix here has no rows of test words. All 36 runs of this notebook are identical, in all epochs and all 4 metrics, to the earlier runs whose matrix also had those rows (W&B), so those rows changed nothing. The numbers come from this run, not from the CPU run on `main`.

In [1]:
# setup: on Colab, connect Google Drive; everywhere, use the GPU if there is one
import os
import sys

# on a GPU, a seed alone does not repeat a run exactly; this variable must be set before the GPU is used
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:2"

import torch

torch.backends.cudnn.deterministic = True
# deterministic operations where they exist, and a warning for the others
torch.use_deterministic_algorithms(True, warn_only=True)

if "google.colab" in sys.modules:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/nlp-project1-art")
    %pip install -q gensim datasets wandb nltk
    # the key comes from Colab Secrets, so it is not in the notebook
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# the W&B server of this project
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Mounted at /content/drive
cuda Tesla T4


Load the token indices, the labels and the embedding matrix that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only evaluated once at the end, in `05_evaluation.ipynb`.

In [2]:
import numpy as np
import torch

data = np.load("features/model2.npz")
# token indices of O1, O2, h1, h2 for every example, and the labels (0 = h1, 1 = h2)
X_train = torch.tensor(data["X_train"]).to(device)
y_train = torch.tensor(data["y_train"]).to(device)
X_valid = torch.tensor(data["X_valid"]).to(device)
y_valid = torch.tensor(data["y_valid"]).to(device)
# GloVe vectors: row i = vector of the word with index i (row 0 = padding)
embedding_matrix = torch.tensor(data["embedding_matrix"]).to(device)
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)
print(embedding_matrix.shape, embedding_matrix.dtype)

torch.Size([16801, 4, 34]) torch.int64
torch.Size([16801]) torch.int64
torch.Size([15840, 300]) torch.float32


## Classifier
The same 2-layer classifier as for model 1 (`03_model1_embeddings.ipynb`, from the course exercise "PyTorch" (Week 1)): linear layer, ReLU, linear layer. In model 2, its input are the final hidden states of the RNN instead of the averaged word vectors.

In [3]:
import torch.nn as nn

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

## Model 2: RNN + classifier
The same LSTM or GRU reads each of the four texts; the final hidden state of its last layer, h_n, represents the text (course slides "Advanced RNNs", slide 16). The four h_n go side by side into the classifier, like the four sentence vectors in model 1. The embedding layer holds the frozen GloVe vectors (`freeze=True`), with index 0 as padding (`padding_idx=0`). `pack_padded_sequence` (course slides "Project Discussions" (Week 4), slide 17) gives the RNN the real length of each text, so h_n is the state after the last real word. Dropout can be added between the RNN layers and on the final hidden states (section "Dropout"); by default it is 0.

In [4]:
from torch.nn.utils.rnn import pack_padded_sequence

class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100):
        super().__init__()
        # embedding layer: the frozen GloVe vectors, row 0 = padding
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        # size of one word vector (300)
        embedding_dim = embedding_matrix.shape[1]
        # 2-layer LSTM or GRU; batch_first=True: input shape [batch size, sequence length, embedding dim]
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        # dropout on the 4 final hidden states before the classifier (0.0 = no dropout)
        self.dropout = nn.Dropout(dropout)
        # classifier on the 4 final hidden states side by side (O1, O2, h1, h2)
        self.classifier = Classifier(4 * hidden_dim, 2, inner_dim=inner_dim)

    def forward(self, x):
        # x: token indices, shape [batch size, 4, 34] (O1, O2, h1, h2)
        features = []
        for i in range(4):
            # one text (O1, O2, h1 or h2) of every example in the batch: [batch size, 34]
            text = x[:, i]
            # real length of each text = number of token indices that are not 0 (padding)
            lengths = (text != 0).sum(dim=1)
            # look up the GloVe vectors: [batch size, 34, 300]
            embedded = self.embedding(text)
            # give the RNN the real lengths, so it stops after the last real word
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            # the RNN reads the text; only the final hidden state is needed
            _, hidden = self.rnn(packed)
            # the LSTM returns (h_n, c_n), the GRU only h_n
            if isinstance(self.rnn, nn.LSTM):
                hidden = hidden[0]
            # h_n of the last layer: [batch size, hidden dim]
            features.append(hidden[-1])
        # the 4 final hidden states side by side: [batch size, 4 * hidden dim]
        features = torch.concat(features, dim=-1)
        # dropout on the final hidden states (active only in training mode)
        features = self.dropout(features)
        return self.classifier(features)

# same start weights every time (as in model 1)
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix).to(device)
print(model)

RNNClassifier(
  (embedding): Embedding(15840, 300, padding_idx=0)
  (rnn): LSTM(300, 128, num_layers=2, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (classifier): Classifier(
    (linear1): Linear(in_features=512, out_features=100, bias=True)
    (relu): ReLU()
    (linear2): Linear(in_features=100, out_features=2, bias=True)
  )
)


Test the untrained model on the first 5 training examples: one pair of logits (h1, h2) per example.

In [5]:
# 5 examples, each with 4 texts of 34 token indices
x = X_train[:5]
# forward pass: one pair of logits per example
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 4, 34])
torch.Size([5, 2])
tensor([[0.0967, 0.0726],
        [0.0921, 0.0704],
        [0.0973, 0.0738],
        [0.0959, 0.0718],
        [0.0912, 0.0662]], device='cuda:0', grad_fn=<AddmmBackward0>)


## Loss, optimizer and batches
The same loss and optimizer as for model 1 (`03_model1_embeddings.ipynb`). The batch size comes from the test in the section "Batch size": 128, where model 1 has 2048.

In [6]:
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

# cross-entropy loss on the 2 logits (applies the softmax itself)
criterion = nn.CrossEntropyLoss()

# the largest batch size that fits into the GPU memory, up to the limit, as in the course slides "Project Discussions"
# (Week 4), slide 14: the longest training example, one forward and backward pass, binary search
def batch_fits(build_model, X, y, index, size):
    try:
        model = build_model()
        x = X[index].repeat(size, *[1] * (X.dim() - 1))
        criterion(model(x), y[:1].repeat(size)).backward()
        del model, x
        torch.cuda.empty_cache()
        return True
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return False

# limit: a batch as large as the training data would make only one update per epoch
def largest_batch_size(build_model, X, y, limit=2048):
    # on a CPU, memory is not the limit: the batch size of the course exercise
    if device.type != "cuda":
        return 32
    # the example with the most real tokens
    index = (X != 0).reshape(len(X), -1).sum(dim=1).argmax()
    if batch_fits(build_model, X, y, index, limit):
        print("Largest batch size:", limit, "(the limit)")
        return limit
    # low fits, high does not; multiples of 32 leave a little memory free
    low, high = 32, limit
    while high - low > 32:
        middle = (low + high) // 2 // 32 * 32
        if batch_fits(build_model, X, y, index, middle):
            low = middle
        else:
            high = middle
    print("Largest batch size:", low)
    return low

MAX_BATCH = largest_batch_size(lambda: RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=256, dropout=0.7).to(device), X_train, y_train)
# training batches, shuffled in every epoch, the last incomplete batch is skipped
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=MAX_BATCH, shuffle=True, drop_last=True)
# equal batches for measuring loss and accuracy (16801 = 53 x 317, 1000 = 10 x 100), not shuffled
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

# number of batches, and the shape of one training batch: batch size, 4 texts, 34 token indices
x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(train_eval_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

Largest batch size: 2048 (the limit)
8 53 10
torch.Size([2048, 4, 34]) torch.Size([2048])


## Evaluation
The same `evaluate()` as for model 1. Check: an untrained model should reach about 0.69 loss and 50% accuracy. Its logit for h1 is slightly higher for every example, so it predicts h1 for all: the share of h1 in the validation set, 508 of 1000 (50.80%).

In [7]:
# same function as in 03_model1_embeddings.ipynb
def evaluate(model, loader):
    # evaluation mode, and no gradients are needed
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            # loss of this batch
            losses.append(criterion(output, y).item())
            # predicted class: the hypothesis with the higher logit
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    # join the batches: one prediction and one label per example
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    # share of correct predictions
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

# untrained model with the same start weights as the training runs
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix).to(device)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6931, accuracy: 50.80%


## Check: overfit one batch
Can the model learn at all? A fresh model is trained on one fixed batch of 32 training examples for 50 steps: the loss must go to almost 0 and the accuracy to 100%; otherwise there is a bug, for example no gradient through the packed sequences ("overfit one batch", Karpathy, 2019). The embedding layer must still hold exactly the GloVe vectors.

In [8]:
# fresh model with the same start weights as the training runs
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix).to(device)
optimizer = Adam(model.parameters(), lr=0.001)
# one fixed batch: the first 32 training examples
x, y = X_train[:32], y_train[:32]

model.train()
for step in range(1, 51):
    # one training step, as for model 1
    output = model(x)
    loss = criterion(output, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # loss and accuracy on the batch every 10 steps
    if step % 10 == 0:
        accuracy = (output.argmax(dim=1) == y).float().mean().item()
        print(f"Step {step}: loss {loss.item():.4f}, accuracy {accuracy:.2%}")

# frozen embeddings: the embedding layer must still be exactly the GloVe matrix
print("Embeddings unchanged:", torch.equal(model.embedding.weight, embedding_matrix))

Step 10: loss 0.6012, accuracy 56.25%
Step 20: loss 0.0390, accuracy 100.00%
Step 30: loss 0.0001, accuracy 100.00%
Step 40: loss 0.0000, accuracy 100.00%
Step 50: loss 0.0000, accuracy 100.00%
Embeddings unchanged: True


The loss on the batch goes down to 0.0000 and the accuracy reaches 100% after 20 steps, and the embedding layer is unchanged: the model can learn, and the GloVe vectors stay frozen.

## Experiment tracking
The training runs are tracked with W&B, as for model 1 (same project `nlp-project1-art-gpu`, and no W&B log lines in the notebook output).

In [9]:
import os

# no W&B log lines in the notebook output; the metrics are still logged
os.environ["WANDB_SILENT"] = "true"

import wandb
wandb.login()


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


True

## Training
The same training function as for model 1 (`03_model1_embeddings.ipynb`). Hyperparameters: the type of RNN (`lstm` or `gru`), its hidden size and the learning rate.

- **Early stopping** (course slides "Project Discussions" (Week 6), slide 38): after 10 epochs without a lower validation loss, the same patience as for model 1, at most 100 epochs, a limit no run reaches. One epoch takes about 4 seconds with batch size 128.
- **Regularization:** no dropout at first, as in model 1; tested in the section "Dropout". Weight decay is tested on the chosen configuration (section "Weight decay").
- **Normalization:** layer normalization is tested once, to limit the number of runs (same slides (Week 7), slide 55), and only in the classifier, because `nn.GRU` has no option for it. It has the lowest validation loss of all candidate runs, so the chosen configuration has it.

In [10]:
import os
import time

os.makedirs("checkpoints", exist_ok=True)

def train(rnn_type="lstm", hidden_dim=128, learning_rate=0.001, dropout=0.0, max_epochs=100, patience=10):
    # run name, e.g. glove-lstm-h128-lr0.001, with dropout e.g. glove-lstm-h128-lr0.001-d0.2
    name = f"glove-{rnn_type}-h{hidden_dim}-lr{learning_rate}"
    if dropout > 0:
        name += f"-d{dropout}"
    checkpoint_path = f"checkpoints/{name}.pth"
    wandb.init(
        project="nlp-project1-art-gpu",
        name=name,
        config={
            "model": f"glove-{rnn_type}",
            "rnn_type": rnn_type,
            "hidden_dim": hidden_dim,
            "num_layers": 2,
            "inner_dim": 100,
            "dropout": dropout,
            "learning_rate": learning_rate,
            "max_epochs": max_epochs,
            "patience": patience,
            "batch_size": train_loader.batch_size,
            "seed": 42,
        },
    )

    # fresh model with the same start weights in every run
    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type=rnn_type, hidden_dim=hidden_dim, dropout=dropout).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        start = time.time()
        # one training step per batch, as for model 1
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        # loss and accuracy after this epoch
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        minutes = (time.time() - start) / 60
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%} ({minutes:.1f} min)")
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                print(f"Early stopping: no lower validation loss in the last {patience} epochs")
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
    # return the model of the best epoch, not the one of the last epoch
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return model

## Batch size
The course recommends the largest batch size that fits into memory (course slides "Project Discussions" (Week 7), slides 41 and 51), but a larger batch makes fewer updates per epoch: 525 with batch size 32, 8 with 2048. Question: **does the largest batch size lower the validation loss?** One sweep tests the configuration chosen in the CPU run with batch size 32, 128, 512 and the largest size, each for at most 1000 epochs, so that early stopping ends a run and not the limit. The learning rate stays at 0.001, so a run does not show the best result possible with its batch size.

**Fixed before the run:** all further runs use the largest tested batch size whose validation loss is within 0.0078 of the lowest one (the standard deviation over the seeds in the CPU run), a trade-off between speed and validation loss. **Result:** 32: 0.5941, 128: 0.5997, 512: 0.6078, 2048: 0.6127. The limit is 0.6019, so all further runs use batch size 128.

In [11]:
# the batch sizes to compare, with the configuration chosen in the CPU run
batch_size_config = {
    "name": "model2-batch-size",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"batch_size": {"values": sorted({32, 128, 512, MAX_BATCH})}},
}

# same training as in sweep_run(), with the training batches of this run
def batch_size_run(hidden_dim=128, learning_rate=0.001, dropout=0.5, max_epochs=1000, patience=10):
    start = time.time()
    # the agent puts the batch size of this run into run.config
    run = wandb.init(project="nlp-project1-art-gpu")
    batch_size = run.config["batch_size"]
    name = f"glove-gru-sweep-h{hidden_dim}-lr{learning_rate}-d{dropout}-b{batch_size}"
    run.name = name
    run.config.update({"model": "glove-gru-sweep", "rnn_type": "gru", "hidden_dim": hidden_dim, "num_layers": 2, "inner_dim": 100, "learning_rate": learning_rate, "dropout": dropout, "max_epochs": max_epochs, "patience": patience, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"
    # training batches with the batch size of this run
    batch_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)

    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in batch_train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    minutes = (time.time() - start) / 60
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%} ({minutes:.1f} min)")

batch_size_id = wandb.sweep(batch_size_config, project="nlp-project1-art-gpu")
wandb.agent(batch_size_id, function=batch_size_run)


Create sweep with ID: 0rqzt6p7
Sweep URL: https://app.forge.coreweave.com/maria-micioni-hochschule-luzern/nlp-project1-art-gpu/sweeps/0rqzt6p7
glove-gru-sweep-h128-lr0.001-d0.5-b32: best epoch 3 of 13, valid loss 0.5941, valid acc 67.50% (2.2 min)
glove-gru-sweep-h128-lr0.001-d0.5-b128: best epoch 6 of 16, valid loss 0.5997, valid acc 66.90% (0.9 min)
glove-gru-sweep-h128-lr0.001-d0.5-b512: best epoch 7 of 17, valid loss 0.6078, valid acc 65.80% (0.5 min)
glove-gru-sweep-h128-lr0.001-d0.5-b2048: best epoch 10 of 20, valid loss 0.6127, valid acc 65.60% (0.5 min)


In [12]:
# the sweep agent leaves its run id in the settings of W&B: without this, the next wandb.init would continue the last sweep run
wandb.teardown()

# the batch size of all further runs, from the test above and the rule fixed before it
BATCH_SIZE = 128
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(len(train_loader), "training batches per epoch, batch size", BATCH_SIZE)

131 training batches per epoch, batch size 128


## Experiments
Hyperparameter tuning on the validation set, as for model 1. Questions:
1. **Does an RNN encoder beat averaging?** Model 1 averages the word vectors (chosen run: validation loss 0.6460, accuracy 64.00%). This needs no extra run.
2. **LSTM or GRU?** Which is better depends on the task and the dataset (course slides "Recurrent Neural Networks" (Week 3), slides 62 and 68).
3. **Is the learning rate 0.01 too high?** It is "the most important hyperparameter" (course slides "Project Discussions" (Week 6), slide 32), and for model 1, 0.01 made most ReLU units inactive.

A grid search (same slides (Week 7), slide 44) trains all 4 combinations of LSTM and GRU with learning rate 0.001 and 0.01, because the best learning rate can differ between them; hidden size 128, 2 layers, no dropout, batch size 128, seed 42. The RNN hidden size and the 100 hidden units of the classifier stay fixed to limit the number of runs; the hidden units changed the scores of model 1 little (all runs that learned: 60.70% to 64.00%). Each configuration is trained once, so small differences can be chance.

In [13]:
# 4 runs: LSTM and GRU, each with learning rate 0.001 and 0.01
for rnn_type in ["lstm", "gru"]:
    for learning_rate in [0.001, 0.01]:
        train(rnn_type=rnn_type, learning_rate=learning_rate)

Epoch 1: train loss 0.6090, train acc 65.97%, valid loss 0.6332, valid acc 63.80% (0.1 min)
Epoch 2: train loss 0.5728, train acc 70.34%, valid loss 0.6145, valid acc 65.80% (0.1 min)
Epoch 3: train loss 0.5261, train acc 73.64%, valid loss 0.6136, valid acc 67.20% (0.1 min)
Epoch 4: train loss 0.4874, train acc 77.04%, valid loss 0.6044, valid acc 67.20% (0.1 min)
Epoch 5: train loss 0.4246, train acc 80.89%, valid loss 0.6172, valid acc 67.40% (0.1 min)
Epoch 6: train loss 0.3742, train acc 84.36%, valid loss 0.6412, valid acc 66.50% (0.1 min)
Epoch 7: train loss 0.2691, train acc 89.63%, valid loss 0.7203, valid acc 67.60% (0.1 min)
Epoch 8: train loss 0.1899, train acc 93.54%, valid loss 0.8012, valid acc 68.20% (0.1 min)
Epoch 9: train loss 0.1340, train acc 95.45%, valid loss 0.9978, valid acc 66.30% (0.1 min)
Epoch 10: train loss 0.1036, train acc 96.28%, valid loss 1.2459, valid acc 66.30% (0.1 min)
Epoch 11: train loss 0.0681, train acc 97.69%, valid loss 1.3635, valid acc 65.

## Dropout
All runs with learning rate 0.001 overfit quickly: the training accuracy reaches 84-87% after 6 epochs, while the validation loss rises after epoch 3 or 4. Question: **does dropout lower the validation loss?**

Dropout (Srivastava et al., 2014; course slides "Preliminaries" (Week 1), slide 20; "Project Discussions" (Week 6), slide 34) is used in two places, not on recurrent connections (Zaremba et al., 2014): between the two RNN layers (argument `dropout`) and on the four final hidden states (`nn.Dropout`), not after the output (slide 34). With dropout 0, `nn.Dropout` changes nothing, so the runs above remain valid.

4 runs: LSTM and GRU with learning rate 0.001 (0.01 collapses the GRU and is not better for the LSTM), each with dropout 0.2 and 0.5.

In [14]:
# 4 runs: LSTM and GRU with learning rate 0.001, each with dropout 0.2 and 0.5
for rnn_type in ["lstm", "gru"]:
    for dropout in [0.2, 0.5]:
        train(rnn_type=rnn_type, learning_rate=0.001, dropout=dropout)

Epoch 1: train loss 0.6190, train acc 64.62%, valid loss 0.6392, valid acc 61.80% (0.1 min)
Epoch 2: train loss 0.5747, train acc 69.54%, valid loss 0.6125, valid acc 65.30% (0.1 min)
Epoch 3: train loss 0.5442, train acc 72.55%, valid loss 0.6114, valid acc 67.00% (0.1 min)
Epoch 4: train loss 0.4968, train acc 76.00%, valid loss 0.6027, valid acc 67.10% (0.1 min)
Epoch 5: train loss 0.4425, train acc 79.80%, valid loss 0.6030, valid acc 67.90% (0.1 min)
Epoch 6: train loss 0.3876, train acc 83.34%, valid loss 0.6250, valid acc 66.10% (0.1 min)
Epoch 7: train loss 0.2952, train acc 87.75%, valid loss 0.7351, valid acc 66.90% (0.1 min)
Epoch 8: train loss 0.2329, train acc 91.49%, valid loss 0.7483, valid acc 67.70% (0.1 min)
Epoch 9: train loss 0.2025, train acc 92.48%, valid loss 0.8654, valid acc 64.60% (0.1 min)
Epoch 10: train loss 0.1445, train acc 94.86%, valid loss 1.0369, valid acc 65.40% (0.1 min)
Epoch 11: train loss 0.1087, train acc 96.78%, valid loss 1.0977, valid acc 64.

Dropout 0.5, the largest value tested, gives the lowest validation loss for both RNNs, so the next larger value, 0.7, is tested too (course slides "Project Discussions" (Week 7), slide 53).

In [15]:
# the next larger dropout, because 0.5 (the largest value tested) was the best
for rnn_type in ["lstm", "gru"]:
    train(rnn_type=rnn_type, learning_rate=0.001, dropout=0.7)

Epoch 1: train loss 0.6607, train acc 60.33%, valid loss 0.6660, valid acc 59.30% (0.1 min)
Epoch 2: train loss 0.6141, train acc 66.37%, valid loss 0.6331, valid acc 64.60% (0.1 min)
Epoch 3: train loss 0.5820, train acc 69.04%, valid loss 0.6160, valid acc 64.60% (0.1 min)
Epoch 4: train loss 0.5514, train acc 71.25%, valid loss 0.6034, valid acc 66.50% (0.1 min)
Epoch 5: train loss 0.5234, train acc 73.96%, valid loss 0.5952, valid acc 65.50% (0.1 min)
Epoch 6: train loss 0.4920, train acc 75.91%, valid loss 0.5975, valid acc 66.70% (0.1 min)
Epoch 7: train loss 0.4641, train acc 78.67%, valid loss 0.5943, valid acc 67.10% (0.1 min)
Epoch 8: train loss 0.4020, train acc 82.31%, valid loss 0.6144, valid acc 67.10% (0.1 min)
Epoch 9: train loss 0.3675, train acc 84.82%, valid loss 0.6359, valid acc 68.00% (0.1 min)
Epoch 10: train loss 0.3219, train acc 87.09%, valid loss 0.6722, valid acc 67.90% (0.1 min)
Epoch 11: train loss 0.2742, train acc 89.31%, valid loss 0.6926, valid acc 67.

## Learning rate 0.0001
The best learning rate of the grid, 0.001, is its smallest value, so the next smaller value, 0.0001, is tested too (course slides "Project Discussions" (Week 7), slides 52 and 53), for the LSTM and the GRU without dropout. The limit of 100 epochs leaves room for the smaller steps (slide 41). The learning rate and the dropout are tuned one after the other, which may miss the best combination (slide 41).

In [16]:
# the next smaller learning rate, without dropout
for rnn_type in ["lstm", "gru"]:
    train(rnn_type=rnn_type, learning_rate=0.0001, max_epochs=100)

Epoch 1: train loss 0.6926, train acc 51.74%, valid loss 0.6928, valid acc 52.10% (0.1 min)
Epoch 2: train loss 0.6509, train acc 61.38%, valid loss 0.6572, valid acc 59.80% (0.1 min)
Epoch 3: train loss 0.6273, train acc 64.91%, valid loss 0.6384, valid acc 62.40% (0.1 min)
Epoch 4: train loss 0.6085, train acc 66.20%, valid loss 0.6298, valid acc 64.20% (0.1 min)
Epoch 5: train loss 0.5955, train acc 67.77%, valid loss 0.6279, valid acc 64.70% (0.1 min)
Epoch 6: train loss 0.5918, train acc 68.20%, valid loss 0.6259, valid acc 65.00% (0.1 min)
Epoch 7: train loss 0.5753, train acc 69.14%, valid loss 0.6195, valid acc 65.40% (0.1 min)
Epoch 8: train loss 0.5624, train acc 70.44%, valid loss 0.6254, valid acc 63.90% (0.1 min)
Epoch 9: train loss 0.5553, train acc 71.23%, valid loss 0.6238, valid acc 64.80% (0.1 min)
Epoch 10: train loss 0.5423, train acc 71.95%, valid loss 0.6257, valid acc 64.40% (0.1 min)
Epoch 11: train loss 0.5292, train acc 73.23%, valid loss 0.6272, valid acc 65.

## Layer normalization
Does layer normalization lower the validation loss of the configuration chosen in the CPU run (GRU, learning rate 0.001, dropout 0.5)? As for model 1, `nn.LayerNorm` goes between the first linear layer of the classifier and the ReLU. `nn.GRU` has no option for it, so this is not the layer normalized RNN of Ba et al. (2016). The check below shows that both runs start from the same weights and get the same first batch. If this run has the lowest validation loss of all candidate runs, it becomes the chosen configuration.

In [17]:
# model 2 with layer norm in the classifier: everything else stays the same
class RNNLayerNormClassifier(RNNClassifier):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100):
        # build the normal model first, so the random starting weights stay the same
        super().__init__(embedding_matrix, rnn_type=rnn_type, hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout, inner_dim=inner_dim)
        # new: linear1 is now followed by layer norm, before the ReLU of the classifier
        self.classifier.linear1 = nn.Sequential(self.classifier.linear1, nn.LayerNorm(inner_dim))


In [18]:
# same seed as in train(): the model with layer norm, then the first batch
torch.manual_seed(42)
model_ln = RNNLayerNormClassifier(embedding_matrix, rnn_type="gru", dropout=0.5).to(device)
x_ln, _ = next(iter(train_loader))

# the same for the model without layer norm
torch.manual_seed(42)
model_plain = RNNClassifier(embedding_matrix, rnn_type="gru", dropout=0.5).to(device)
x_plain, _ = next(iter(train_loader))

# same starting weights in the RNN and the classifier, and same first batch?
print(torch.equal(model_ln.rnn.weight_ih_l0, model_plain.rnn.weight_ih_l0))
print(torch.equal(model_ln.classifier.linear1[0].weight, model_plain.classifier.linear1.weight))
print(torch.equal(model_ln.classifier.linear2.weight, model_plain.classifier.linear2.weight))
print(torch.equal(x_ln, x_plain))
# how many parameters does layer norm add?
print(sum(p.numel() for p in model_ln.parameters()) - sum(p.numel() for p in model_plain.parameters()))


True
True
True
True
200


In [19]:
import time

# the chosen configuration of model 2, with layer norm in the classifier
rnn_type, hidden_dim, learning_rate, dropout, max_epochs, patience = "gru", 128, 0.001, 0.5, 100, 10
name = f"glove-{rnn_type}-ln-h{hidden_dim}-lr{learning_rate}-d{dropout}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art-gpu",
    name=name,
    config={
        "model": f"glove-{rnn_type}-ln",
        "rnn_type": rnn_type,
        "hidden_dim": hidden_dim,
        "num_layers": 2,
        "inner_dim": 100,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "max_epochs": max_epochs,
        "patience": patience,
        "batch_size": train_loader.batch_size,
        "seed": 42,
    },
)

# same training as in train(), with the layer norm class
torch.manual_seed(42)
model = RNNLayerNormClassifier(embedding_matrix, rnn_type=rnn_type, hidden_dim=hidden_dim, dropout=dropout).to(device)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")
# number of epochs in a row without a new lowest validation loss
epochs_without_improvement = 0

for epoch in range(max_epochs):
    start = time.time()
    model.train()
    for x, y in train_loader:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader)
    valid_loss, valid_acc = evaluate(model, valid_loader)
    minutes = (time.time() - start) / 60
    print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%} ({minutes:.1f} min)")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print(f"Early stopping: no lower validation loss in the last {patience} epochs")
            break

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
print("For comparison, the same settings without layer norm: best epoch 6, valid loss 0.5997, valid acc 66.90%")

Epoch 1: train loss 0.6878, train acc 56.26%, valid loss 0.6894, valid acc 54.10% (0.1 min)
Epoch 2: train loss 0.6112, train acc 66.20%, valid loss 0.6214, valid acc 66.40% (0.1 min)
Epoch 3: train loss 0.5833, train acc 69.17%, valid loss 0.6124, valid acc 67.10% (0.1 min)
Epoch 4: train loss 0.5564, train acc 70.91%, valid loss 0.6088, valid acc 67.60% (0.1 min)
Epoch 5: train loss 0.5177, train acc 74.43%, valid loss 0.5962, valid acc 68.10% (0.1 min)
Epoch 6: train loss 0.4906, train acc 77.17%, valid loss 0.5867, valid acc 66.70% (0.1 min)
Epoch 7: train loss 0.4477, train acc 79.96%, valid loss 0.6057, valid acc 66.80% (0.1 min)
Epoch 8: train loss 0.3923, train acc 83.74%, valid loss 0.6131, valid acc 67.70% (0.1 min)
Epoch 9: train loss 0.3388, train acc 86.40%, valid loss 0.6507, valid acc 66.00% (0.1 min)
Epoch 10: train loss 0.2882, train acc 89.36%, valid loss 0.6952, valid acc 66.00% (0.1 min)
Epoch 11: train loss 0.2315, train acc 91.80%, valid loss 0.7344, valid acc 67.

## Results
Load the best checkpoint of every model 2 run, evaluate it on the validation set, and sort the results by the validation loss (lowest first), as for model 1.

In [20]:
# all model 2 runs: RNN type, learning rate, dropout
runs = [(rnn_type, learning_rate, dropout) for rnn_type in ["lstm", "gru"]
        for learning_rate, dropout in [(0.0001, 0.0), (0.001, 0.0), (0.01, 0.0), (0.001, 0.2), (0.001, 0.5), (0.001, 0.7)]]

results = []
for rnn_type, learning_rate, dropout in runs:
    # same run name as in train()
    name = f"glove-{rnn_type}-h128-lr{learning_rate}"
    if dropout > 0:
        name += f"-d{dropout}"
    # build the model with the settings of the run and load the weights of its best epoch
    model = RNNClassifier(embedding_matrix, rnn_type=rnn_type, dropout=dropout).to(device)
    model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
    valid_loss, valid_acc = evaluate(model, valid_loader)
    results.append((valid_loss, valid_acc, name))

# the run with layer norm: the same evaluation, with its own class
name = "glove-gru-ln-h128-lr0.001-d0.5"
model = RNNLayerNormClassifier(embedding_matrix, rnn_type="gru", dropout=0.5).to(device)
model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
valid_loss, valid_acc = evaluate(model, valid_loader)
results.append((valid_loss, valid_acc, name))

# lowest validation loss first
for valid_loss, valid_acc, name in sorted(results):
    print(f"{name}: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")

glove-gru-ln-h128-lr0.001-d0.5: valid loss 0.5867, valid acc 66.70%
glove-lstm-h128-lr0.001-d0.7: valid loss 0.5943, valid acc 67.10%
glove-gru-h128-lr0.001-d0.7: valid loss 0.5981, valid acc 65.90%
glove-lstm-h128-lr0.001-d0.5: valid loss 0.5989, valid acc 67.80%
glove-gru-h128-lr0.001-d0.5: valid loss 0.5997, valid acc 66.90%
glove-lstm-h128-lr0.001-d0.2: valid loss 0.6027, valid acc 67.10%
glove-gru-h128-lr0.001: valid loss 0.6033, valid acc 68.20%
glove-lstm-h128-lr0.001: valid loss 0.6044, valid acc 67.20%
glove-lstm-h128-lr0.01: valid loss 0.6052, valid acc 66.20%
glove-gru-h128-lr0.001-d0.2: valid loss 0.6083, valid acc 67.20%
glove-lstm-h128-lr0.0001: valid loss 0.6195, valid acc 65.40%
glove-gru-h128-lr0.0001: valid loss 0.6202, valid acc 66.10%
glove-gru-h128-lr0.01: valid loss 0.6930, valid acc 50.80%


## Sweep
The experiments above tuned the learning rate and the dropout one after the other and kept the RNN hidden size fixed, which may miss the best setting (course slides "Project Discussions" (Week 7), slide 41). Question: **does another combination of RNN hidden size, learning rate and dropout lower the validation loss?** A W&B sweep (slide 57) trains all 12 combinations: RNN hidden size 64, 128 and 256, learning rate 1e-3 and 3.16e-4 (slide 52), dropout 0.5 and 0.7. Only the GRU is used, with 100 hidden units in the classifier, to limit the number of runs (slide 55): with the same settings, the LSTM and the GRU differ by at most 0.0056 where both learn.

The method is `grid`, as for model 1. Every run has seed 42, patience 10 and at most 100 epochs. Two combinations (RNN hidden size 128, learning rate 1e-3, dropout 0.5 and 0.7) repeat earlier runs and must give the same best epoch and validation loss. The selection rule stays: the run with the lowest validation loss of all candidate runs is chosen.

In [21]:
# the sweep: all combinations of these values (3 x 2 x 2 = 12 runs)
sweep_config = {
    "name": "model2-grid",
    "method": "grid",
    # the value in the run summary by which W&B compares the runs
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {
        "hidden_dim": {"values": [64, 128, 256]},
        "learning_rate": {"values": [0.001, 0.000316]},
        "dropout": {"values": [0.5, 0.7]},
    },
}

# one run of the sweep: the W&B agent calls this function once per combination
def sweep_run(max_epochs=100, patience=10):
    start = time.time()
    # the agent puts the values of this combination into run.config
    run = wandb.init(project="nlp-project1-art-gpu")
    hidden_dim = run.config["hidden_dim"]
    learning_rate = run.config["learning_rate"]
    dropout = run.config["dropout"]
    # run name as for the other runs, and the settings that are the same in all runs
    name = f"glove-gru-sweep-h{hidden_dim}-lr{learning_rate}-d{dropout}"
    run.name = name
    run.config.update({"model": "glove-gru-sweep", "rnn_type": "gru", "num_layers": 2, "inner_dim": 100, "max_epochs": max_epochs, "patience": patience, "batch_size": train_loader.batch_size, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"

    # same training as in train()
    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    minutes = (time.time() - start) / 60
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%} ({minutes:.1f} min)")


In [22]:
# register the sweep at W&B and train all its combinations (a grid ends by itself)
sweep_id = wandb.sweep(sweep_config, project="nlp-project1-art-gpu")
wandb.agent(sweep_id, function=sweep_run)


Create sweep with ID: csebjbe6
Sweep URL: https://app.forge.coreweave.com/maria-micioni-hochschule-luzern/nlp-project1-art-gpu/sweeps/csebjbe6
glove-gru-sweep-h64-lr0.001-d0.5: best epoch 5 of 15, valid loss 0.5959, valid acc 66.40% (0.9 min)
glove-gru-sweep-h64-lr0.000316-d0.5: best epoch 11 of 21, valid loss 0.5994, valid acc 66.90% (1.2 min)
glove-gru-sweep-h128-lr0.001-d0.5: best epoch 6 of 16, valid loss 0.5997, valid acc 66.90% (1.0 min)
glove-gru-sweep-h128-lr0.000316-d0.5: best epoch 6 of 16, valid loss 0.6075, valid acc 66.30% (1.0 min)
glove-gru-sweep-h256-lr0.001-d0.5: best epoch 5 of 15, valid loss 0.5978, valid acc 66.70% (1.0 min)
glove-gru-sweep-h256-lr0.000316-d0.5: best epoch 7 of 17, valid loss 0.6089, valid acc 65.60% (1.1 min)
glove-gru-sweep-h64-lr0.001-d0.7: best epoch 7 of 17, valid loss 0.5966, valid acc 67.70% (1.0 min)
glove-gru-sweep-h64-lr0.000316-d0.7: best epoch 13 of 23, valid loss 0.5991, valid acc 66.40% (1.3 min)
glove-gru-sweep-h128-lr0.001-d0.7: best

In [23]:
# load the best checkpoint of every sweep run and sort by the validation loss, as in the section "Results"
sweep_values = sweep_config["parameters"]
results = []
for hidden_dim in sweep_values["hidden_dim"]["values"]:
    for learning_rate in sweep_values["learning_rate"]["values"]:
        for dropout in sweep_values["dropout"]["values"]:
            name = f"glove-gru-sweep-h{hidden_dim}-lr{learning_rate}-d{dropout}"
            model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout).to(device)
            model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
            valid_loss, valid_acc = evaluate(model, valid_loader)
            results.append((valid_loss, valid_acc, name))

for valid_loss, valid_acc, name in sorted(results):
    print(f"{name}: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")


glove-gru-sweep-h256-lr0.001-d0.7: valid loss 0.5946, valid acc 67.70%
glove-gru-sweep-h64-lr0.001-d0.5: valid loss 0.5959, valid acc 66.40%
glove-gru-sweep-h64-lr0.001-d0.7: valid loss 0.5966, valid acc 67.70%
glove-gru-sweep-h256-lr0.001-d0.5: valid loss 0.5978, valid acc 66.70%
glove-gru-sweep-h128-lr0.001-d0.7: valid loss 0.5981, valid acc 65.90%
glove-gru-sweep-h64-lr0.000316-d0.7: valid loss 0.5991, valid acc 66.40%
glove-gru-sweep-h64-lr0.000316-d0.5: valid loss 0.5994, valid acc 66.90%
glove-gru-sweep-h128-lr0.001-d0.5: valid loss 0.5997, valid acc 66.90%
glove-gru-sweep-h256-lr0.000316-d0.7: valid loss 0.6039, valid acc 66.40%
glove-gru-sweep-h128-lr0.000316-d0.7: valid loss 0.6061, valid acc 66.80%
glove-gru-sweep-h128-lr0.000316-d0.5: valid loss 0.6075, valid acc 66.30%
glove-gru-sweep-h256-lr0.000316-d0.5: valid loss 0.6089, valid acc 65.60%


## Other seeds
Question: **how much do the validation scores of the chosen configuration change with the seed?** It is trained with seeds 1 to 4 (course slides "Project Discussions" (Week 7), slide 42). The seed is not tuned (slide 56): the chosen checkpoint stays the one with seed 42, and these runs are only reported.

In [24]:
from torch.optim import AdamW

seed_config = {
    "name": "model2-seeds",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"seed": {"values": [1, 2, 3, 4]}},
}

# one run: as in sweep_run(), with the chosen configuration and the seed or the weight decay of this run
def chosen_run(hidden_dim=128, learning_rate=0.001, dropout=0.5, max_epochs=100, patience=10):
    start = time.time()
    run = wandb.init(project="nlp-project1-art-gpu")
    seed = run.config.get("seed", 42)
    weight_decay = run.config.get("weight_decay", 0.0)
    kind = f"wd{weight_decay}" if weight_decay > 0 else f"seed{seed}"
    name = f"glove-gru-ln-{kind}-h{hidden_dim}-lr{learning_rate}-d{dropout}"
    run.name = name
    run.config.update({"model": "glove-gru-ln-wd" if weight_decay > 0 else "glove-gru-ln-seeds", "seed": seed, "weight_decay": weight_decay, "rnn_type": "gru", "hidden_dim": hidden_dim, "num_layers": 2, "inner_dim": 100, "learning_rate": learning_rate, "dropout": dropout, "max_epochs": max_epochs, "patience": patience, "batch_size": train_loader.batch_size})
    checkpoint_path = f"checkpoints/{name}.pth"

    # the seed of this run
    torch.manual_seed(seed)
    model = RNNLayerNormClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout).to(device)
    # AdamW only with weight decay: without it, the two make the same update steps
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay) if weight_decay > 0 else Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            best_train_acc = train_acc
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    minutes = (time.time() - start) / 60
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}, train acc at the best epoch {best_train_acc:.2%}, last valid loss {valid_loss:.4f} ({minutes:.1f} min)")

seed_id = wandb.sweep(seed_config, project="nlp-project1-art-gpu")
wandb.agent(seed_id, function=chosen_run)

Create sweep with ID: ba620w1t
Sweep URL: https://app.forge.coreweave.com/maria-micioni-hochschule-luzern/nlp-project1-art-gpu/sweeps/ba620w1t
glove-gru-ln-seed1-h128-lr0.001-d0.5: best epoch 7 of 17, valid loss 0.5922, valid acc 69.00%, train acc at the best epoch 79.74%, last valid loss 1.1047 (1.0 min)
glove-gru-ln-seed2-h128-lr0.001-d0.5: best epoch 6 of 16, valid loss 0.5964, valid acc 67.10%, train acc at the best epoch 77.37%, last valid loss 1.0648 (1.0 min)
glove-gru-ln-seed3-h128-lr0.001-d0.5: best epoch 5 of 15, valid loss 0.6005, valid acc 67.20%, train acc at the best epoch 73.28%, last valid loss 1.0131 (0.9 min)
glove-gru-ln-seed4-h128-lr0.001-d0.5: best epoch 6 of 16, valid loss 0.5953, valid acc 66.40%, train acc at the best epoch 76.33%, last valid loss 1.0224 (1.0 min)


In [25]:
# validation loss and accuracy of all five seeds, from the saved checkpoints
names = ["glove-gru-ln-h128-lr0.001-d0.5"] + [f"glove-gru-ln-seed{seed}-h128-lr0.001-d0.5" for seed in [1, 2, 3, 4]]
scores = []
for name in names:
    model = RNNLayerNormClassifier(embedding_matrix, rnn_type="gru", hidden_dim=128, dropout=0.5).to(device)
    model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
    scores.append(evaluate(model, valid_loader))
losses, accuracies = zip(*scores)
# ddof=1: sample standard deviation of the five runs
print(f"chosen configuration: valid loss {np.mean(losses):.4f} (sd {np.std(losses, ddof=1):.4f}, seeds {[round(float(v), 4) for v in losses]}), valid acc {np.mean(accuracies):.2%} (sd {np.std(accuracies, ddof=1):.2%}, seeds {[round(v * 100, 2) for v in accuracies]})")

chosen configuration: valid loss 0.5942 (sd 0.0051, seeds [0.5867, 0.5922, 0.5964, 0.6005, 0.5953]), valid acc 67.28% (sd 1.01%, seeds [66.7, 69.0, 67.1, 67.2, 66.4])


## Weight decay
The chosen model overfits: its validation loss rises after epoch 6. Question: **does weight decay reduce the overfitting and lower the validation loss?** The chosen configuration, with its layer norm and dropout, is trained with AdamW instead of Adam, because the two only differ with weight decay (course slides "Project Discussions" (Week 6), slide 31): 0.001, 0.01, the default of `torch.optim.AdamW`, and 0.1, in exponential steps (same slides (Week 7), slide 52). Weight decay should be "not larger than gradient * learning rate" (slide 34); 0.1 only marks the edge of the range. Seed 42: the same start weights and batches as the chosen run. **Fixed before these runs:** they are a report, not candidates, and the chosen run stays, because one run cannot be separated from the spread over the seeds.

In [26]:
# the chosen configuration with weight decay
weight_decay_config = {
    "name": "model2-weight-decay",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"weight_decay": {"values": [0.001, 0.01, 0.1]}},
}
weight_decay_id = wandb.sweep(weight_decay_config, project="nlp-project1-art-gpu")
wandb.agent(weight_decay_id, function=chosen_run)

Create sweep with ID: x48u0zcl
Sweep URL: https://app.forge.coreweave.com/maria-micioni-hochschule-luzern/nlp-project1-art-gpu/sweeps/x48u0zcl
glove-gru-ln-wd0.001-h128-lr0.001-d0.5: best epoch 6 of 16, valid loss 0.5918, valid acc 66.90%, train acc at the best epoch 77.13%, last valid loss 1.0621 (1.0 min)
glove-gru-ln-wd0.01-h128-lr0.001-d0.5: best epoch 6 of 16, valid loss 0.5921, valid acc 66.60%, train acc at the best epoch 77.41%, last valid loss 1.1214 (1.0 min)
glove-gru-ln-wd0.1-h128-lr0.001-d0.5: best epoch 6 of 16, valid loss 0.5897, valid acc 65.80%, train acc at the best epoch 76.87%, last valid loss 1.0030 (1.0 min)


## Model 1 vs. model 2
Both chosen models are scored on the same 1000 validation examples, so they are compared with the exact McNemar test (McNemar, 1947) on the examples that only one of them gets right.

In [27]:
from scipy.stats import binomtest

# the chosen model 1: its input (the averaged GloVe vectors) and the weights of its best epoch
X1_valid = torch.tensor(np.load("features/model1.npz")["X_valid"]).to(device)
model1 = Classifier(1200, 2, inner_dim=600).to(device)
model1.load_state_dict(torch.load("checkpoints/glove-mlp-sweep-h600-lr0.000316-b2048.pth", weights_only=True))
# the chosen model 2
model2 = RNNLayerNormClassifier(embedding_matrix, rnn_type="gru", dropout=0.5).to(device)
model2.load_state_dict(torch.load("checkpoints/glove-gru-ln-h128-lr0.001-d0.5.pth", weights_only=True))
# evaluation mode: no dropout
model1.eval()
model2.eval()

with torch.no_grad():
    # True where the prediction is right
    correct1 = model1(X1_valid).argmax(dim=1) == y_valid
    correct2 = model2(X_valid).argmax(dim=1) == y_valid
# & = both True, ~ = not
only_model1 = (correct1 & ~correct2).sum().item()
only_model2 = (~correct1 & correct2).sum().item()
print("both right:", (correct1 & correct2).sum().item())
print("only model 1 right:", only_model1)
print("only model 2 right:", only_model2)
print("both wrong:", (~correct1 & ~correct2).sum().item())
# exact McNemar test: binomial test on the examples that only one model gets right
print(f"p = {binomtest(only_model1, only_model1 + only_model2, 0.5).pvalue:.4f}")

both right: 507
only model 1 right: 133
only model 2 right: 160
both wrong: 200
p = 0.1286


## Check: do the observations help?
In `03_model1_embeddings.ipynb`, model 1 reached a lower validation loss with only h1 and h2. Do the chosen models gain from the observations? Each validation example keeps its h1 and h2 but gets O1 and O2 of a random validation example, 10 times (permutation feature importance, Breiman, 2001). If a model gains from the real observations, its accuracy falls. The models are not retrained, so this tests the trained models, not what a model could learn.

In [28]:
# same random swaps in every run of this cell
torch.manual_seed(42)
changed1, changed2, accuracies1, accuracies2 = [], [], [], []
with torch.no_grad():
    # predictions with the real observations
    predictions1 = model1(X1_valid).argmax(dim=1)
    predictions2 = model2(X_valid).argmax(dim=1)
    for repeat in range(10):
        # for every example, the position of a random validation example
        other = torch.randperm(len(y_valid))
        # model 1: the first 600 of the 1200 numbers are O1 and O2
        swapped1 = X1_valid.clone()
        swapped1[:, :600] = X1_valid[other, :600]
        # model 2: the first 2 of the 4 texts are O1 and O2
        swapped2 = X_valid.clone()
        swapped2[:, :2] = X_valid[other, :2]
        # predictions with the observations of other examples
        new1 = model1(swapped1).argmax(dim=1)
        new2 = model2(swapped2).argmax(dim=1)
        # how many predictions change, and the accuracy
        changed1.append((new1 != predictions1).sum().item())
        changed2.append((new2 != predictions2).sum().item())
        accuracies1.append((new1 == y_valid).float().mean().item())
        accuracies2.append((new2 == y_valid).float().mean().item())
print(f"model 1: accuracy {(predictions1 == y_valid).float().mean().item():.2%} -> {np.mean(accuracies1):.2%} (lowest {min(accuracies1):.2%}, highest {max(accuracies1):.2%}), {np.mean(changed1):.0f} of 1000 predictions change")
print(f"model 2: accuracy {(predictions2 == y_valid).float().mean().item():.2%} -> {np.mean(accuracies2):.2%} (lowest {min(accuracies2):.2%}, highest {max(accuracies2):.2%}), {np.mean(changed2):.0f} of 1000 predictions change")

model 1: accuracy 64.00% -> 61.40% (lowest 60.40%, highest 63.40%), 226 of 1000 predictions change
model 2: accuracy 66.70% -> 67.21% (lowest 66.50%, highest 67.80%), 60 of 1000 predictions change


## Check: does the position of the hypotheses matter?
If a prediction depended only on the content of the texts, swapping h1 and h2 would swap it on every example. The validation examples are scored again with h1 and h2 swapped, without retraining. `04b_per_choice.ipynb` tests scoring each hypothesis with the same classifier, which removes this dependence.

In [29]:
with torch.no_grad():
    # model 1: O1, O2, then h2 before h1 (h1 = numbers 600 to 899, h2 = 900 to 1199)
    swapped1 = torch.concat([X1_valid[:, :600], X1_valid[:, 900:], X1_valid[:, 600:900]], dim=1)
    # model 2: the texts in the order O1, O2, h2, h1
    swapped2 = X_valid[:, [0, 1, 3, 2]]
    # predictions after the swap: class 0 now means h2, class 1 now means h1
    new1 = model1(swapped1).argmax(dim=1)
    new2 = model2(swapped2).argmax(dim=1)
# same class as before the swap = the prediction did not follow the hypothesis
print(f"model 1: {(new1 == predictions1).sum().item()} of 1000 predictions do not follow the swap, accuracy after the swap {(new1 == 1 - y_valid).float().mean().item():.2%}")
print(f"model 2: {(new2 == predictions2).sum().item()} of 1000 predictions do not follow the swap, accuracy after the swap {(new2 == 1 - y_valid).float().mean().item():.2%}")


model 1: 399 of 1000 predictions do not follow the swap, accuracy after the swap 62.90%
model 2: 124 of 1000 predictions do not follow the swap, accuracy after the swap 66.70%


## Conclusion
- **Chosen:** GRU with layer norm in the classifier, RNN hidden size 128, learning rate 0.001, dropout 0.5, batch size 128, best epoch 6: validation loss 0.5867, accuracy 66.70%, the lowest validation loss of all 29 candidate runs. RNN hidden size, learning rate and dropout lie inside the tested ranges.
- **RNN or averaging:** all 28 RNN runs that learned are below the best candidate run of model 1 (0.5867 to 0.6202 vs. 0.6460). On the chosen runs, the exact McNemar test shows no significant difference in correct predictions (66.70% vs. 64.00%, only model 2 right 160, only model 1 right 133, p = 0.1286). Over 5 seeds, every model 2 seed is ahead of every model 1 seed: validation loss 0.5867 to 0.6005 vs. 0.6451 to 0.6471, accuracy 66.40% to 69.00% vs. 62.80% to 64.00%. The gap remains with the same learning rate and no dropout: GRU 0.6033, LSTM 0.6044, model 1 0.6530 (batch size 128 vs. 2048). The experiments do not separate word order from the other differences between an RNN and averaging.
- **LSTM or GRU:** similar. The LSTM is lower in 5 of 6 settings, by at most 0.0056 where both learn. The sweep and layer norm only use the GRU, so a better LSTM setting may exist.
- **Learning rate:** 0.01 collapses the GRU (50.80%, h1 for all examples) and is not better for the LSTM (0.6052); 0.0001 is worse than 0.001 (GRU 0.6202 vs. 0.6033).
- **Dropout:** 0.5 lowers the validation loss (LSTM 0.6044 to 0.5989, GRU 0.6033 to 0.5997), and 0.7, the boundary of the tested range, lowers it further (LSTM 0.5943, GRU 0.5981).
- **Overfitting:** the checkpoint is from epoch 6 (training accuracy 77.17%, validation accuracy 66.70%); by epoch 16, the training accuracy is 97.55% and the validation loss 1.0707. Dropout slows this (last validation loss of the GRU: 2.2543 without dropout, 1.3826 with 0.5, 1.0268 with 0.7), but a gap remains at the best epoch.
- **Weight decay** (a report, not candidates): no lower validation loss than the chosen run (0.001: 0.5918, 0.01: 0.5921, 0.1: 0.5897, without: 0.5867), and the overfitting stays: all stop at the same epochs, with a training accuracy of about 77% at the best epoch and a last validation loss of 1.00 to 1.12 (without: 1.07). One seed.
- **Layer normalization** (classifier only): at seed 42, a lower validation loss than the same settings without it (0.5867 vs. 0.5997), about the same accuracy (66.70% vs. 66.90%). Over 5 seeds, the two are not separated: 0.5942 (sd 0.0051) vs. 0.5963 (sd 0.0045), 67.28% vs. 66.34%. The seeds without it are from the first GPU run of this notebook (W&B runs `glove-gru-seed1` to `glove-gru-seed4`), whose matrix also had the rows of the test words, which no training example uses. Tested with these settings only.
- **Other seeds:** validation loss 0.5942 on average (sd 0.0051, 0.5867 to 0.6005), accuracy 67.28% (sd 1.01%). Seed 42 has the lowest validation loss of the five, so 0.5867 is optimistic, and the chosen configuration is not clearly the best one.
- **Sweep:** no combination is below the chosen run (0.5946 to 0.6089). With one seed per configuration, the others are not shown to be worse. The two repeated runs give the same results.
- **Batch size:** larger batches have a higher validation loss at these settings (32: 0.5941, 128: 0.5997, 512: 0.6078, 2048: 0.6127). Batch size 128 is a trade-off by the rule fixed before the test; it does not show that the batch sizes perform equally.
- **CPU run:** the settings chosen in the CPU run, with batch size 32 and seed 42, reach 0.5872 on the laptop and 0.5941 here. The CPU and GPU pipelines differ in preprocessing and training settings (vocabulary, patience), so this comparison does not isolate the effect of hardware.
- **Observations:** with the observations of other validation examples, model 1 falls from 64.00% to 61.40% (226 of 1000 predictions change), while model 2 does not fall (66.70% to 67.21%, 60 change). So the trained model 2 shows no gain from the real observations on the validation set.
- **Position of the hypotheses:** swapping h1 and h2 should swap the prediction, but 399 of 1000 predictions of model 1 and 124 of model 2 do not follow the swap. Accuracy after the swap: 62.90% and 66.70%. So this comparison applies to the fixed-position format only.
- **Input format:** model 1 with per-choice scoring (`04b_per_choice.ipynb`, one seed) reaches 0.5858 and 68.90%, at least level with the chosen model 2 (0.5867 and 66.70%). So the gap between the models may depend on the input format.

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier` (same as in `03_model1_embeddings.ipynb`), based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Model 2: 2-layer RNN (LSTM or GRU, PyTorch implementations) with the frozen word embeddings of model 1 and a 2-layer classifier with ReLU, as required in the Project 1 description; `nn.Embedding.from_pretrained` with `freeze=True` and `padding_idx=0` from the PyTorch documentation `torch.nn.Embedding`, https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html; `nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers)` as in the course exercise "Sequence-to-sequence RNN" (Week 4); `nn.LSTM`, `nn.GRU` and `dropout` (between the RNN layers) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html and https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html; `batch_first=True` for the input shape [batch size, sequence length, embedding dim] from the course slides "Preliminaries" (Week 1), slide 14; final hidden states of the texts as input of the classifier from the course slides "Advanced RNNs" (Week 4), slides 16-17.
- Forward pass of model 2: `pack_padded_sequence` for padding in RNNs from the course slides "Project Discussions" (Week 4), slide 17, with the lengths on the CPU and `enforce_sorted=False` from the PyTorch documentation `torch.nn.utils.rnn.pack_padded_sequence`, https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html; final hidden state of the last layer `hn[-1]` and `torch.concat(..., dim=-1)` as in the course exercise "Sequence-to-sequence RNN" (Week 4); output `(h_n, c_n)` of the LSTM and `h_n` of the GRU from the PyTorch documentation `torch.nn.LSTM` and `torch.nn.GRU`; `isinstance` from the Python documentation, https://docs.python.org/3/library/functions.html#isinstance; counting the non-zero indices with `(text != 0).sum(dim=1)` (PyTorch documentation `torch.sum`, https://docs.pytorch.org/docs/stable/generated/torch.sum.html); test forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss, optimizer and batches: the same as in `03_model1_embeddings.ipynb`: cross-entropy loss from the course slides "Preliminaries" (Week 1), slide 27, and `nn.CrossEntropyLoss` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html; the term "logits" from the course slides "Project Discussions" (Week 5), slide 27; Adam from the course slides "Preliminaries" (Week 1), slide 29; `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` and `drop_last=True` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; `next(iter(...))` as in the PyTorch tutorial "Datasets & DataLoaders", https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html.
- Evaluation: the same function as in `03_model1_embeddings.ipynb` (batch losses, `argmax` predictions and labels as in the validation of the course exercise "Experiment Tracking"; `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).
- Check with one batch: "overfit one batch" from Karpathy, A. (2019), *A Recipe for Training Neural Networks*, https://karpathy.github.io/2019/04/25/recipe/; training step as in the course exercise "PyTorch" (Week 1); `torch.equal` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.equal.html.
- Experiment tracking and training: the same as in `03_model1_embeddings.ipynb`: `wandb.login()`, `wandb.init(project=..., config=...)` and `wandb.finish()` as in the course exercise "Experiment Tracking"; logging with `epoch` as x-axis as in the W&B documentation "Log objects and media", https://docs.wandb.ai/models/track/log; best values in the run summary as in the W&B documentation "Log summary metrics", https://docs.wandb.ai/models/track/log/log-summary; training loop as in the PyTorch tutorial "Optimizing Model Parameters"; saving and loading only the parameters (`state_dict`) as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html, which is the recommended way on slide 36 of the course slides "Project Discussions" (Week 6). Early stopping on the validation loss with a patience from the course slides "Project Discussions" (Week 6), slide 38. Time per epoch with `time.time()` from the Python documentation, https://docs.python.org/3/library/time.html.
- Experiments: hyperparameter tuning on the validation set, as on slide 10 of the course slides "Course Project HS26"; LSTM vs. GRU from the course slides "Recurrent Neural Networks" (Week 3), slides 62 and 68; the learning rate as "the most important hyperparameter" from the course slides "Project Discussions" (Week 6), slide 32.
- Dropout: dropout probability as a hyperparameter from the course slides "Preliminaries" (Week 1), slides 19-20; dropout and normalization as regularization, and dropout not after the output, from the course slides "Project Discussions" (Week 6), slide 34; dropout from Srivastava, Hinton, Krizhevsky, Sutskever & Salakhutdinov (2014), *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*, Journal of Machine Learning Research 15, https://jmlr.org/papers/v15/srivastava14a.html; dropout only on the connections between the layers of an LSTM (not from one time step to the next) from Zaremba, Sutskever & Vinyals (2014), *Recurrent Neural Network Regularization*, https://arxiv.org/abs/1409.2329; `nn.Dropout` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html; argument `dropout` of `nn.LSTM` and `nn.GRU` ("a Dropout layer on the outputs of each LSTM layer except the last layer") from the PyTorch documentation `torch.nn.LSTM`.
- Grid search and learning rate 0.0001: grid search (all combinations) from the course slides "Project Discussions" (Week 7), slide 44; testing beyond the boundary of the tested range (slide 53), exponentially changing values (slide 52), a sensible number of epochs and sequential tuning (slide 41), and the number of runs depending on the budget (slide 55) from the same slides.
- Results and conclusion: loading the checkpoints with `load_state_dict` as in the PyTorch tutorial "Save and Load the Model" and sorting with `sorted()` as in `03_model1_embeddings.ipynb` (Python documentation "Sorting Techniques", https://docs.python.org/3/howto/sorting.html); LSTM vs. GRU from the course slides "Recurrent Neural Networks" (Week 3), slide 68.
- Model 1 vs. model 2: exact McNemar test from McNemar, Q. (1947), *Note on the sampling error of the difference between correlated proportions or percentages*, Psychometrika 12(2), 153–157, https://doi.org/10.1007/BF02295996, as a binomial test on the examples that only one model gets right ("McNemar's test", Wikipedia, https://en.wikipedia.org/wiki/McNemar%27s_test), with `scipy.stats.binomtest` from SciPy (Virtanen et al., 2020), https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html; `&` and `~` on True/False tensors from the PyTorch documentation `torch.bitwise_and` and `torch.bitwise_not`, https://docs.pytorch.org/docs/stable/generated/torch.bitwise_and.html and https://docs.pytorch.org/docs/stable/generated/torch.bitwise_not.html; loading the checkpoints as in the section "Results"; `model.eval()` and `torch.no_grad()` as in `evaluate()`.
- Check of the observations: permutation feature importance from Breiman, L. (2001), *Random Forests*, Machine Learning 45(1), 5–32, https://doi.org/10.1023/A:1010933404324, and the scikit-learn documentation "Permutation feature importance", https://scikit-learn.org/stable/modules/permutation_importance.html; `torch.randperm`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.randperm.html; `Tensor.clone`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.Tensor.clone.html; `min()` and `max()`, Python documentation, https://docs.python.org/3/library/functions.html; origin of the splits: Bhagavatula et al. (2020), Table 6, and course slides "Course Project HS26", slide 8.
- Layer normalization: `nn.LayerNorm`, https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html, `nn.Sequential`, https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html, and `nn.GRU` (no option for layer normalization), https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html, from the PyTorch documentation; before the nonlinearity and the layer normalized RNN from Ba, Kiros & Hinton (2016), *Layer Normalization*, https://arxiv.org/abs/1607.06450; normalization as an option and layer norm (yes/no) as an experiment from the course slides "Project Discussions" (Week 6), slide 34, and (Week 7), slide 44; subclass from the Python tutorial "Classes", https://docs.python.org/3/tutorial/classes.html#inheritance. Check: `torch.numel` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.numel.html; `weight_ih_l0`, the variable `weight_ih_l[k]` in the PyTorch documentation `torch.nn.GRU`, https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html.
- GPU: `torch.device`, `torch.cuda.is_available` and `Tensor.to`, PyTorch documentation "CUDA semantics", https://docs.pytorch.org/docs/stable/notes/cuda.html; `torch.cuda.empty_cache` and `torch.cuda.OutOfMemoryError`, https://docs.pytorch.org/docs/stable/cuda.html; `Tensor.repeat`, https://docs.pytorch.org/docs/stable/generated/torch.Tensor.repeat.html; `lambda`, Python tutorial, https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions. HSLU GPUHub and Google Colab from the course slides "Course Project HS26", slide 11. The largest batch size, found with the longest example, one forward and backward pass and binary search, from the course slides "Project Discussions" (Week 4), slide 14, and (Week 7), slide 51. Environment variables on Colab: Google Colab Secrets and `drive.mount`, https://colab.research.google.com/. Repeated runs on a GPU: `torch.backends.cudnn.deterministic`, https://docs.pytorch.org/docs/stable/backends.html, and PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html; `torch.use_deterministic_algorithms` (`warn_only`), https://docs.pytorch.org/docs/stable/generated/torch.use_deterministic_algorithms.html; `CUBLAS_WORKSPACE_CONFIG`, from the warning about RNN functions in `torch.nn.LSTM`, https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html. Setup on Colab: `sys.modules`, https://docs.python.org/3/library/sys.html#sys.modules, and `os.chdir`, https://docs.python.org/3/library/os.html#os.chdir, Python documentation; `%pip`, IPython documentation, https://ipython.readthedocs.io/en/stable/interactive/magics.html.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md; all code reviewed and understood by me.
- Sweep: W&B sweeps from the course slides "Project Discussions" (Week 7), slide 57, and the W&B documentation on the sweep configuration, https://docs.coreweave.com/models/sweeps/define-sweep-configuration, its options (`method`, `metric`, `parameters`, `values`), https://docs.coreweave.com/models/sweeps/sweep-config-keys, and the sweeps walkthrough (`wandb.sweep`, `wandb.agent` with `function`, `wandb.init` inside the function, `run.config`), https://docs.coreweave.com/models/sweeps/walkthrough; `run.config.update`, W&B documentation on the run config, https://docs.coreweave.com/models/track/config; `run.name`, W&B documentation `Run`, https://docs.coreweave.com/models/ref/python/experiments/run; grid search (slide 44), sequential tuning (slide 41), finer steps such as 1e-3.5 (slide 52) and the number of trials (slide 55) from the same slides.
- Batch size: the largest batch size that fits into memory from the course slides "Project Discussions" (Week 7), slides 41 and 51; training batches with `DataLoader` as in the section "Loss, optimizer and batches". `sorted`, https://docs.python.org/3/library/functions.html#sorted; `wandb.teardown`, W&B documentation, https://docs.coreweave.com/models/ref/python/functions/teardown.
- Weight decay: Adam and AdamW (Week 6, slide 31), weight decay "not larger than gradient * learning rate" (Week 6, slide 34) and exponentially changing values (Week 7, slide 52) from the course slides "Project Discussions"; `torch.optim.AdamW` (`weight_decay`, default 0.01), https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html, from Loshchilov & Hutter (2019); `dict.get`, Python documentation, https://docs.python.org/3/library/stdtypes.html#dict.get; `Config` with `get` and `update`, W&B documentation on the run config, https://docs.coreweave.com/models/track/config.
- Other seeds: reporting the average of several runs and the debate about tuning the seed from the course slides "Project Discussions" (Week 7), slides 42 and 56; `np.std` with `ddof`, NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.std.html; `zip(*scores)`, Python documentation, https://docs.python.org/3/library/functions.html#zip.
- Check of the position: `torch.concat` as in the section "Model 2: RNN + classifier"; indexing with a list of positions, NumPy documentation "Indexing on ndarrays", https://numpy.org/doc/stable/user/basics.indexing.html, which works the same on PyTorch tensors.
- Output of W&B: `WANDB_SILENT` (no W&B log lines in the notebook output), W&B documentation "Environment variables", https://docs.coreweave.com/models/track/environment-variables; `os.environ`, Python documentation, https://docs.python.org/3/library/os.html#os.environ.
- Input format check: `04b_per_choice.ipynb` (see the sources there).